# EDA courte — HPP Lean (admission → HPPsev)

Objectif : vérifier le taux de positifs, les NA, et rappeler l’anti-leakage temporel.  
Le preprocessing / modèle sont dans `src/train.py` (pas ici).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import FEATURES, TARGET, PROCESSED_DIR, DEFAULT_RAW_CSV
from src.prepare_data import prepare

processed = PROCESSED_DIR / "admission_hpp.csv"
if processed.exists():
    df = pd.read_csv(processed)
else:
    df = prepare(DEFAULT_RAW_CSV)

print(df.shape)
df.head()

In [ ]:
vc = df[TARGET].value_counts(dropna=False)
rate = df[TARGET].mean()
print(vc)
print(f"Taux {TARGET}+ : {100 * rate:.2f} %")

vc.plot(kind="bar", title=f"Distribution {TARGET}", color=["#4C78A8", "#E45756"])
plt.xticks(rotation=0)
plt.ylabel("Effectif")
plt.show()

In [ ]:
na = df[FEATURES].isna().mean().sort_values(ascending=False)
print(na.head(10).to_string())
na.plot(kind="barh", figsize=(8, 6), title="% NA par feature")
plt.xlabel("% manquant")
plt.tight_layout()
plt.show()

## Anti-leakage (à dire à l’oral)

- Features = infos **admission / pré-accouchement** uniquement (`src/config.py`).
- Exclus : transfusion, embolisation, hystérectomie, scores néonataux, etc. (post-événement).
- Split **stratifié** + Imputer / Scaler / OHE **fit sur le train seulement** (`Pipeline` dans `train.py`).
- Seuil τ calibré sur **validation**, métriques annoncées sur **test** une seule fois.